## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("../../src").resolve()))
from condpred import report
from condpred.experiments import e2

SMOKE = os.environ.get("CONDPRED_SMOKE") == "1"
OUT = Path("../../notebooks/tables/condpred/E2" + ("/smoke" if SMOKE else ""))
FIG = OUT / "figures"
CACHE = "E2/smoke" if SMOKE else "E2"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

SPEC = e2.build_initial_spec(SMOKE)
TRAJ = {}

## 2. Write spec

In [ ]:
report.write_spec(SPEC, OUT, modules=e2.E2_MODULES)
e2.verify_spec_immutable(OUT, SPEC, e2.E2_MODULES)

## 3. Leakage test

In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal(600)
leaks = e2.run_leakage_e2(x, rng)
ldf = pd.DataFrame(leaks, columns=["pipeline", "pass"])
report.save_table(ldf, OUT, "leakage")
assert ldf["pass"].all(), ldf[~ldf["pass"]]

## 4. Simulate (cached)

In [ ]:
TRAJ = e2.load_trajectories(SPEC, CACHE)

## 5. Fit fixed models on libraries; oracle

## 6. Losses

## 7. Features

## 8. Loss prediction A

## 9. Detection B: lag curves and delays

## 10. Primary decision

In [ ]:
RESULT = e2.run_full_pipeline(SPEC, OUT, FIG, cache_prefix=CACHE)

## 11. Secondary

In [ ]:
secondary = pd.read_csv(OUT / "results_secondary_1.csv")
assert "holm_p" in secondary.columns
assert {"wz", "gz"} <= set(secondary["geometry"])
assert 50 in set(secondary["W"].astype(int))
assert {"ridge", "lgbm"} <= set(secondary["predictor"])
assert {"M1a", "M1c", "M1b"} <= set(secondary["model"])
display(secondary.head())

## 12. Figures and tables

In [ ]:
decision = pd.read_csv(OUT / "decision.csv")
lag_curve = pd.read_csv(OUT / "lag_curve.csv")
assert set(lag_curve["bin"].unique()) == set(e2.LAG_BIN_LABELS), lag_curve["bin"].unique()
assert {"M0", "WRQA", "LRQA", "M1a", "M1b"} <= set(lag_curve["model"])
oracle = pd.read_csv(OUT / "oracle.csv")
assert oracle["spearman_oracle"].notna().any()
display(decision)
display(oracle)
display(lag_curve.head())